In [1]:
import requests
import time
import json
from bs4 import BeautifulSoup

# ----------------------
# PARAMETERS
# ----------------------
STACK_KEYWORDS = "stress"
STACK_SITE = "stackoverflow"
STACK_PAGE_SIZE = 30
STACK_MAX_PAGES = 2

GITHUB_KEYWORDS = "stress"
GITHUB_PER_PAGE = 30
GITHUB_TOKEN = ""

OUTPUT_FILE = "stress_posts_clean.json"

# Stress-related keywords
KEYWORDS = [
    "stress", "burnout", "overwork", "overburden", "workload",
    "pressure", "anxiety", "tired", "exhausted", "work environment",
]

# ----------------------
# UTILITIES
# ----------------------
def clean_html(raw_html):
    if not raw_html:
        return ""
    soup = BeautifulSoup(raw_html, "html.parser")
    return soup.get_text().strip()

def matches_keywords(text):
    text = text.lower()
    return any(k in text for k in KEYWORDS)

# ----------------------
# STACKOVERFLOW DATA
# ----------------------
def fetch_stackoverflow():
    posts = []

    for page in range(1, STACK_MAX_PAGES + 1):

        url = "https://api.stackexchange.com/2.3/search/advanced"
        params = {
            "q": STACK_KEYWORDS,
            "site": STACK_SITE,
            "pagesize": STACK_PAGE_SIZE,
            "page": page,
        }

        res = requests.get(url, params=params).json()
        items = res.get("items", [])

        for q in items:
            qid = q["question_id"]

            # Fetch the body
            q_url = f"https://api.stackexchange.com/2.3/questions/{qid}"
            q_res = requests.get(q_url, params={"site": STACK_SITE, "filter": "withbody"}).json()

            if "items" not in q_res or not q_res["items"]:
                continue

            question = q_res["items"][0]

            title = question["title"]
            body = clean_html(question.get("body", ""))

            combined_text = f"{title}\n\n{body}"

            if matches_keywords(combined_text):
                posts.append({
                    "platform": "stackoverflow",
                    "text": combined_text
                })

            time.sleep(0.2)

    return posts

# ----------------------
# GITHUB DATA
# ----------------------
def fetch_github():
    posts = []

    headers = {"Accept": "application/vnd.github+json"}
    if GITHUB_TOKEN:
        headers["Authorization"] = f"token {GITHUB_TOKEN}"

    url = "https://api.github.com/search/issues"
    params = {
        "q": f"{GITHUB_KEYWORDS} in:title,body is:issue",
        "per_page": GITHUB_PER_PAGE
    }

    res = requests.get(url, headers=headers, params=params).json()
    items = res.get("items", [])

    for item in items:
        title = item["title"]
        body = clean_html(item.get("body", "") or "")

        combined_text = f"{title}\n\n{body}"

        if matches_keywords(combined_text):
            posts.append({
                "platform": "github",
                "text": combined_text
            })

    return posts

# ----------------------
# RUN
# ----------------------
if __name__ == "__main__":
    stack = fetch_stackoverflow()
    github = fetch_github()

    all_posts = stack + github

    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        json.dump(all_posts, f, ensure_ascii=False, indent=2)

    print(f"Saved {len(all_posts)} clean posts to {OUTPUT_FILE}")


Saved 90 clean posts to stress_posts_clean.json
